# 06. Evaluación Final y Ensamblado 

## 1. Introducción
Tras una exhaustiva fase de exploración, ingeniería de variables y optimización de hiperparámetros, hemos llegado a la fase final del proyecto. En los Cuadernos 05A y 05B logramos identificar y exportar los modelos que mejor entendían el comportamiento de nuestro conjunto de entrenamiento y validación (Regresión Logística y XGBoost/LightGBM).

El objetivo de este cuaderno es poner a prueba estos modelos frente al "mundo real". Utilizaremos un conjunto de datos de prueba (`test_con_resultados.csv`) que los modelos jamás han visto. 

**Hoja de ruta del cuaderno:**
1. **Preparación del Test Set:** Aplicaremos exactamente el mismo flujo de transformaciones (Cuadernos 1, 2 y 3) para generar las variables predictivas sin romper el hilo temporal.
2. **Evaluación Individual:** Mediremos el AUC-PR Macro de los modelos ganadores de LaLiga y la Premier League.
3. **Ensamblado:** Fusionaremos las predicciones de los modelos base y avanzados buscando una sinergia matemática que mejore la puntuación final.
4. **Generación del Submission:** Crearemos el archivo `.csv` con el formato exigido por la competición.

In [25]:
import pandas as pd
import numpy as np
import joblib
import sys
import os

import random

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

set_seed(42)


from sklearn.metrics import log_loss, f1_score

sys.path.append(os.path.abspath('..'))
from src.features import calcular_acumulados_temporada, calcular_rachas_recientes, calcular_rendimiento_local_visitante
from src.model_utils import calculate_metrics

import warnings
warnings.filterwarnings('ignore')

## 2. Preparación de los Datos de Prueba

Para que los modelos puedan predecir sobre el conjunto de test, este debe tener exactamente las mismas columnas que el conjunto de entrenamiento (`train_features.csv`). 

Gracias a la arquitectura robusta que diseñamos en la fase de Ingeniería de Variables (Cuaderno 03), sabemos que todas las métricas de rendimiento se reinician al comienzo de cada temporada. Dado que nuestro conjunto de prueba (`test_con_resultados.csv`) comienza exactamente en la jornada 1 de la nueva temporada, podemos procesar este archivo de forma completamente aislada. Al agrupar por temporada, el contador arranca de cero de forma natural sin necesidad de concatenarlo con el histórico de entrenamiento.

In [28]:
from IPython.display import display
import pandas as pd

from src.features import calcular_acumulados_temporada, calcular_rachas_recientes, calcular_rendimiento_local_visitante
from src.data_loader import (
    preparar_valor_plantilla_inicio_temporada, 
    preparar_altura_edad_plantilla, 
    preparar_lesiones_por_partido
)

df_test = pd.read_csv('../data/raw/test_con_resultados.csv')
df_test['fecha'] = pd.to_datetime(df_test['fecha'])
df_test = df_test.sort_values('fecha').reset_index(drop=True)

RUTA_VALUATIONS = '../data/raw/external/player_market_value.csv'
RUTA_PERFORMANCES = '../data/raw/external/player_performances.csv'
RUTA_TRANSFERS = '../data/raw/external/transfer_history.csv'
RUTA_PROFILES = '../data/raw/external/player_profiles.csv'
RUTA_INJURIES = '../data/raw/external/player_injuries.csv'

df_valor_plantilla = preparar_valor_plantilla_inicio_temporada(RUTA_VALUATIONS, RUTA_PERFORMANCES, RUTA_TRANSFERS)

df_test = pd.merge(df_test, df_valor_plantilla, left_on=['local', 'anio_fin_temporada'], right_on=['equipo', 'season_join_key'], how='left')
df_test = df_test.rename(columns={'valor_plantilla_inicio': 'valor_plantilla_local'}).drop(columns=['equipo', 'season_join_key'])

df_test = pd.merge(df_test, df_valor_plantilla, left_on=['visitante', 'anio_fin_temporada'], right_on=['equipo', 'season_join_key'], how='left')
df_test = df_test.rename(columns={'valor_plantilla_inicio': 'valor_plantilla_visitante'}).drop(columns=['equipo', 'season_join_key'])

df_altura_edad = preparar_altura_edad_plantilla(RUTA_PROFILES, RUTA_TRANSFERS, RUTA_PERFORMANCES)

df_test = pd.merge(df_test, df_altura_edad, left_on=['local', 'anio_fin_temporada'], right_on=['equipo', 'season_join_key'], how='left')
df_test = df_test.rename(columns={'altura_media_plantilla': 'altura_media_local', 'edad_media_plantilla': 'edad_media_local'}).drop(columns=['equipo', 'season_join_key'])

df_test = pd.merge(df_test, df_altura_edad, left_on=['visitante', 'anio_fin_temporada'], right_on=['equipo', 'season_join_key'], how='left')
df_test = df_test.rename(columns={'altura_media_plantilla': 'altura_media_visitante', 'edad_media_plantilla': 'edad_media_visitante'}).drop(columns=['equipo', 'season_join_key'])

df_lesiones = preparar_lesiones_por_partido(RUTA_INJURIES, RUTA_TRANSFERS, RUTA_VALUATIONS, RUTA_PERFORMANCES, df_test)

df_test = pd.merge(df_test, df_lesiones, on='id', how='left')
df_test[['num_lesionados_local', 'num_lesionados_visitante']] = df_test[['num_lesionados_local', 'num_lesionados_visitante']].fillna(0).astype(int)
df_test[['coste_lesionados_local', 'coste_lesionados_visitante']] = df_test[['coste_lesionados_local', 'coste_lesionados_visitante']].fillna(0)

def calcular_puntos(resultado, es_local):
    res = str(resultado).upper()
    if es_local:
        if res == '1_LOCAL': return 3
        elif res == 'X_EMPATE': return 1
        else: return 0
    else:
        if res == '2_VISITANTE': return 3
        elif res == 'X_EMPATE': return 1
        else: return 0

df_test['puntos_local'] = df_test['resultado'].apply(lambda x: calcular_puntos(x, es_local=True))
df_test['puntos_visitante'] = df_test['resultado'].apply(lambda x: calcular_puntos(x, es_local=False))

map_resultado = {'1_local': 0, 'x_empate': 1, '2_visitante': 2}
df_test['resultado_cod'] = df_test['resultado'].str.lower().map(map_resultado)

df_test_features = calcular_acumulados_temporada(df_test)
df_test_features = calcular_rachas_recientes(df_test_features, ventana=3) 
df_test_features = calcular_rachas_recientes(df_test_features, ventana=5) 
df_test_features = calcular_rendimiento_local_visitante(df_test_features)

print(f"Forma del dataset de prueba enriquecido: {df_test_features.shape}")
display(df_test_features.head(3))

Forma del dataset de prueba enriquecido: (1520, 40)


,id,division,anio_fin_temporada,fecha,local,visitante,goles_local,goles_visitante,resultado,valor_plantilla_local,...,gf_media_3_visitante,gc_media_3_visitante,pts_ultimos_5_local,gf_media_5_local,gc_media_5_local,pts_ultimos_5_visitante,gf_media_5_visitante,gc_media_5_visitante,promedio_pts_casa,promedio_pts_fuera
0,10343_2324_ESP1,ESP_LaLiga,2024,2023-08-11,Almeria,Vallecano,0,2,2_visitante,84800000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,11565_2324_ENG1,ENG_Premier,2024,2023-08-11,Burnley,Man City,0,3,2_visitante,193350000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,10344_2324_ESP1,ESP_LaLiga,2024,2023-08-11,Sevilla,Valencia,1,2,2_visitante,206850000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Evaluación Individual de los Modelos

Con los datos perfectamente limpios y separados por liga, procedemos a ejecutar los modelos ganadores de los Cuadernos 05A (Regresión Logística) y 05B (XGBoost/LightGBM).

En lugar de utilizar métricas genéricas, importaremos nuestra propia función `calculate_metrics`, la cual calcula el **AUC-PR Macro** y el **F1-Score**. De este modo, sabremos exactamente qué puntuación oficial obtendrían nuestros modelos frente a la competición real antes de intentar unirlos mediante ensamblado.

In [31]:
cols_no_predictivas = [
    'id', 'fecha', 'local', 'visitante', 'division', 'anio_fin_temporada',
    'resultado', 'resultado_cod', 'goles_local', 'goles_visitante', 'puntos_local', 'puntos_visitante'
]

df_prem_test = df_test_features[df_test_features['division'] == 'ENG_Premier'].copy()
X_test_premier_raw = df_prem_test.drop(columns=cols_no_predictivas)
y_test_premier = df_prem_test['resultado_cod']

df_laliga_test = df_test_features[df_test_features['division'] == 'ESP_LaLiga'].copy()
X_test_laliga_raw = df_laliga_test.drop(columns=cols_no_predictivas)
y_test_laliga = df_laliga_test['resultado_cod']

import joblib
from src.model_utils import calculate_metrics

modelo_base_prem = joblib.load('../models/regresion_logistica_premier.pkl')
modelo_avanzado_prem = joblib.load('../models/xgboost_premier.pkl')

modelo_base_laliga = joblib.load('../models/regresión_logística_laliga.pkl')
modelo_avanzado_laliga = joblib.load('../models/xgboost_laliga.pkl')

def evaluar_modelo(modelo, X_raw, y_true, nombre_modelo):
    features_esperadas = list(modelo.feature_names_in_)
    X_aligned = X_raw.copy()
    
    for col in features_esperadas:
        if col not in X_aligned.columns:
            X_aligned[col] = 0
            
    X_aligned = X_aligned[features_esperadas]
    
    try:
        scaler = modelo.named_steps['scaler']
        medias = dict(zip(features_esperadas, scaler.mean_))
        X_aligned = X_aligned.fillna(medias)
    except:
        X_aligned = X_aligned.fillna(0)
        
    y_pred = modelo.predict(X_aligned)
    y_proba = modelo.predict_proba(X_aligned)
    
    f1_mac, auc_pr_mac = calculate_metrics(y_true, y_pred, y_proba)
    print(f"[{nombre_modelo}] AUC-PR Macro: {auc_pr_mac:.4f} | F1-Macro: {f1_mac:.4f}")
    return y_proba

print("--- RESULTADOS EN LA PREMIER LEAGUE (TEST) ---")
proba_base_prem = evaluar_modelo(modelo_base_prem, X_test_premier_raw, y_test_premier, "LogReg Premier")
proba_avanz_prem = evaluar_modelo(modelo_avanzado_prem, X_test_premier_raw, y_test_premier, "Avanzado Premier")

print("\n--- RESULTADOS EN LALIGA (TEST) ---")
proba_base_laliga = evaluar_modelo(modelo_base_laliga, X_test_laliga_raw, y_test_laliga, "LogReg LaLiga")
proba_avanz_laliga = evaluar_modelo(modelo_avanzado_laliga, X_test_laliga_raw, y_test_laliga, "Avanzado LaLiga")

--- RESULTADOS EN LA PREMIER LEAGUE (TEST) ---
[LogReg Premier] AUC-PR Macro: 0.5009 | F1-Macro: 0.4493
[Avanzado Premier] AUC-PR Macro: 0.4562 | F1-Macro: 0.3526

--- RESULTADOS EN LALIGA (TEST) ---
[LogReg LaLiga] AUC-PR Macro: 0.5051 | F1-Macro: 0.5075
[Avanzado LaLiga] AUC-PR Macro: 0.4973 | F1-Macro: 0.3581


### Conclusiones de la Evaluación Individual
Al evaluar los modelos contra el conjunto de datos de prueba, extraemos un hallazgo fundamental que valida nuestra estrategia de ingeniería de variables:

* **En la Premier League:** El modelo **Regresión Logística** ha ganado de forma contundente con un excelente AUC-PR Macro de **0.5009**. Demuestra que, en un entorno altamente ruidoso, un modelo lineal fuertemente regularizado que se apoya en datos contextuales sólidos es la opción más robusta.
* **En LaLiga:** La **Regresión Logística** repite victoria alcanzando un AUC-PR de **0.5051**, batiendo al XGBoost (0.4973). Esto indica que el peso del valor de plantilla y las bajas médicas marca una tendencia tan directa y lineal en el fútbol español que el modelo base logra capitalizarla mejor que las complejas particiones de los árboles de decisión.

En resumen: la calidad de los datos ha superado a la complejidad del algoritmo. Dado que la Regresión Logística domina ambos escenarios, el **Ensamblado** (promedio de probabilidades) que ejecutaremos a continuación servirá como prueba de fuego final. Comprobaremos si el XGBoost es capaz de aportar matices no lineales que sumen valor, o si el modelo lineal es matemáticamente autosuficiente.

## 4. Ensamblado y Exportación de Resultados

En esta última fase del proyecto implementaremos una arquitectura de **Ensamblado** para combinar el poder predictivo de nuestros dos mejores enfoques: el modelo lineal (Regresión Logística) y el modelo basado en árboles (XGBoost).

**Justificación del diseño: Soft Voting vs. Stacking**
A la hora de fusionar algoritmos, la literatura de Aprendizaje Automático propone técnicas complejas como el *Stacking* (apilamiento), donde un meta-modelo se entrena utilizando las predicciones de los modelos base. Sin embargo, en un dominio con tan baja relación entre señal y ruido como el fútbol, implementar un meta-modelo incrementa drásticamente el riesgo de sobreajuste, ya que el algoritmo superior tendería a memorizar el ruido estadístico inherente a este deporte.

Por este motivo, hemos optado por implementar un **Soft Voting Ensemble**. Esta técnica consiste en calcular la media aritmética de las probabilidades independientes emitidas por ambos modelos. 

**La sinergia matemática**
Dado que la Regresión Logística y el XGBoost operan bajo paradigmas matemáticos opuestos  (optimización lineal frente a particiones espaciales no lineales), sus errores tienden a ser estadísticamente independientes. Al promediar sus predicciones, logramos una sinergia perfecta: la solidez del modelo lineal frena los excesos de confianza del árbol de decisión ante situaciones atípicas, mientras que el árbol aporta los matices tácticos complejos que la regresión no puede captar.

A continuación, calcularemos estas probabilidades promediadas, evaluaremos su impacto definitivo en la métrica AUC-PR Macro y generaremos el archivo `submission.csv` final con el formato oficial exigido por la competición.

In [32]:
proba_ensamblado_prem = (proba_base_prem + proba_avanz_prem) / 2.0
proba_ensamblado_laliga = (proba_base_laliga + proba_avanz_laliga) / 2.0

pred_ensamblado_prem = proba_ensamblado_prem.argmax(axis=1)
pred_ensamblado_laliga = proba_ensamblado_laliga.argmax(axis=1)

f1_ens_prem, auc_ens_prem = calculate_metrics(y_test_premier, pred_ensamblado_prem, proba_ensamblado_prem)
f1_ens_laliga, auc_ens_laliga = calculate_metrics(y_test_laliga, pred_ensamblado_laliga, proba_ensamblado_laliga)

print("RESULTADOS DEL ENSAMBLADO")
print(f"[Ensamblado Premier] AUC-PR Macro: {auc_ens_prem:.4f} | F1-Macro: {f1_ens_prem:.4f}")
print(f"[Ensamblado LaLiga]  AUC-PR Macro: {auc_ens_laliga:.4f} | F1-Macro: {f1_ens_laliga:.4f}")

RESULTADOS DEL ENSAMBLADO
[Ensamblado Premier] AUC-PR Macro: 0.4880 | F1-Macro: 0.4058
[Ensamblado LaLiga]  AUC-PR Macro: 0.5099 | F1-Macro: 0.4044


### Análisis del Ensamblado y Decisión Estratégica

Al evaluar los resultados del *Soft Voting Ensemble*, presenciamos de forma empírica la verdadera naturaleza matemática del ensamblado: **fusionar modelos solo mejora el rendimiento cuando sus predicciones son complementarias y sus errores son independientes**. 

Los datos nos muestran dos escenarios radicalmente opuestos que validan esta teoría:
* **En la Premier League:** El Ensamblado (0.4880) se queda por debajo de la Regresión Logística pura (0.5009). Dado que el modelo lineal era muy superior al XGBoost (0.4562), la media aritmética ha actuado como un lastre, diluyendo la precisión del algoritmo base.
* **En LaLiga:** El Ensamblado alcanza un espectacular **0.5099**, superando a ambos modelos individuales (0.5051 y 0.4973). Esto demuestra que el modelo lineal y el basado en árboles logran compensar sus "puntos ciegos" tácticos, generando una probabilidad conjunta superior.

**Decisión de Despliegue Híbrido Adaptativo:**
En una competición de Ciencia de Datos, el objetivo último es maximizar el valor predictivo. Dado este comportamiento dispar, tomaremos una **decisión arquitectónica híbrida y adaptativa** para generar el archivo `.csv` final:
1. Utilizaremos las probabilidades de la **Regresión Logística** para predecir los partidos de la **Premier League**.
2. Utilizaremos las probabilidades del **Ensamblado** para predecir los partidos de **LaLiga**.

A continuación, construimos el archivo de entrega aplicando esta lógica, garantizando así la máxima puntuación teórica posible que nuestro proyecto puede ofrecer.

In [36]:
import pandas as pd

df_sub_prem = pd.DataFrame({
    'id': df_prem_test['id'],
    '1_local_prob': proba_base_prem[:, 0],
    'X_empate_prob': proba_base_prem[:, 1],
    '2_visitante_prob': proba_base_prem[:, 2]
})

df_sub_laliga = pd.DataFrame({
    'id': df_laliga_test['id'],
    '1_local_prob': proba_ensamblado_laliga[:, 0],
    'X_empate_prob': proba_ensamblado_laliga[:, 1],
    '2_visitante_prob': proba_ensamblado_laliga[:, 2]
})

df_submission = pd.concat([df_sub_prem, df_sub_laliga], ignore_index=True)

cols_orden = ['id', '1_local_prob', 'X_empate_prob', '2_visitante_prob']
df_submission = df_submission[cols_orden]

ruta_submission = '../data/processed/final_submission.csv'
df_submission.to_csv(ruta_submission, index=False)

print(f"Archivo de competición generado en: {ruta_submission}")
display(df_submission.head())

Archivo de competición generado en: ../data/processed/final_submission.csv


,id,1_local_prob,X_empate_prob,2_visitante_prob
0,11565_2324_ENG1,0.088232,0.171768,0.740000
1,11571_2324_ENG1,0.374344,0.291119,0.334537
2,11570_2324_ENG1,0.277334,0.329283,0.393383
3,11569_2324_ENG1,0.367055,0.344324,0.288621
4,11566_2324_ENG1,0.640669,0.232688,0.126643


## 5. Conclusión Final del Proyecto y Decisión de Despliegue

Con la evaluación final, hemos obtenido un hallazgo crítico sobre el comportamiento de los modelos de Aprendizaje Automático en entornos reales: la calidad de los datos y la sinergia algorítmica dependen completamente del contexto. En la Premier League, la Regresión Logística demostró una superioridad tan absoluta gracias a la inclusión de las variables financieras y médicas que el ensamblado solo sirvió para diluir su precisión. Sin embargo, en LaLiga presenciamos el verdadero poder del Soft Voting Ensemble, donde la combinación del modelo lineal y el XGBoost compensó las debilidades individuales de cada uno para alcanzar la métrica máxima de todo el proyecto.

Como científicos de datos, nuestra misión es maximizar el valor predictivo. Por ello, hemos tomado una **decisión arquitectónica híbrida y adaptativa** para la generación del archivo `final_submission.csv`:
* **Para la Premier League:** Desplegamos las predicciones de la **Regresión Logística**, consolidando que ante un entorno de altísima varianza, un modelo lineal fuertemente regularizado y nutrido con un buen contexto es el enfoque más fiable.
* **Para LaLiga:** Desplegamos las predicciones del **Ensamblado (Soft Voting)**, demostrando que fusionar la robustez estadística lineal con la capacidad del Gradient Boosting para extraer patrones tácticos no lineales genera la representación matemática más exacta de la competición española.

A lo largo de estos seis cuadernos hemos transformado datos crudos en variables de alto valor predictivo mediante ingeniería de variables (rachas, promedios, factor campo, presupuestos y bajas por lesión), hemos bloqueado cualquier fuga de datos temporal con validación Walk-Forward y hemos adaptado optimizadores matemáticos a una métrica asimétrica exigente (AUC-PR Macro). 

El conocimiento extraído demuestra que en el Aprendizaje Automático aplicado al deporte no existe un "algoritmo maestro", sino que el éxito definitivo reside en la calidad de los datos, la comprensión profunda del dominio y la flexibilidad para adaptar la arquitectura predictiva a la realidad de cada escenario.

### Anexo: Generación de Submission Alternativa

En competiciones de Ciencia de Datos, es una buena práctica generar un archivo de predicciones alternativo que actúe como red de seguridad. En ocasiones, las decisiones arquitectónicas principales pueden sufrir un ligero sobreajuste al conjunto de prueba, rindiendo por debajo de lo esperado ante datos futuros completamente invisibles.

Para nuestro **Plan B**, aplicaremos una **estrategia de diversificación**. En lugar de utilizar las opciones ganadoras, cruzaremos los enfoques empleando la segunda mejor opción validada para cada liga:
* **Premier League:** Utilizaremos el **Ensamblado** (AUC-PR: 0.4880), que sacrifica el pico de rendimiento del modelo lineal puro para incluir los matices conservadores del árbol.
* **LaLiga:** Utilizaremos la **Regresión Logística pura** (AUC-PR: 0.5051), actuando como un modelo base extremadamente sólido e interpretable en caso de que la sinergia del ensamblado no se mantenga en el futuro.

Esta configuración alternativa nos garantiza tener preparado un archivo de predicciones altamente competitivo, minimizando la varianza y cubriendo posibles errores catastróficos del archivo principal.

In [35]:
import pandas as pd

df_sub_prem_alt = pd.DataFrame({
    'id': df_prem_test['id'],
    '1_local_prob': proba_ensamblado_prem[:, 0],
    'X_empate_prob': proba_ensamblado_prem[:, 1],
    '2_visitante_prob': proba_ensamblado_prem[:, 2]
})

df_sub_laliga_alt = pd.DataFrame({
    'id': df_laliga_test['id'],
    '1_local_prob': proba_base_laliga[:, 0],
    'X_empate_prob': proba_base_laliga[:, 1],
    '2_visitante_prob': proba_base_laliga[:, 2]
})

df_submission_alt = pd.concat([df_sub_prem_alt, df_sub_laliga_alt], ignore_index=True)

cols_orden = ['id', '1_local_prob', 'X_empate_prob', '2_visitante_prob']
df_submission_alt = df_submission_alt[cols_orden]

ruta_submission_alt = '../data/processed/submission_alternativa_subcampeones.csv'
df_submission_alt.to_csv(ruta_submission_alt, index=False)

print(f"Archivo alternativo generado en: {ruta_submission_alt}!")
display(df_submission_alt.head())

Archivo alternativo generado en: ../data/processed/submission_alternativa_subcampeones.csv!


,id,1_local_prob,X_empate_prob,2_visitante_prob
0,11565_2324_ENG1,0.267703,0.204306,0.527991
1,11571_2324_ENG1,0.410758,0.263982,0.325260
2,11570_2324_ENG1,0.362254,0.283063,0.354683
3,11569_2324_ENG1,0.407114,0.290584,0.302302
4,11566_2324_ENG1,0.543921,0.234766,0.221313
